<a href="https://colab.research.google.com/github/gloriabee/DEMovies/blob/main/bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download the source data file from kaggle

In [1]:
import kagglehub
path = kagglehub.dataset_download("tiagoadrianunes/imdb-top-5000-movies")

Using Colab cache for faster access to the 'imdb-top-5000-movies' dataset.


## Import necessary libraries

In [2]:
import os
import shutil
from pyspark.sql import SparkSession

## Extract data from file with PySpark

In [3]:
spark= SparkSession.builder.getOrCreate()
movies_df=spark.read.csv(os.path.join(path, "results_with_crew.csv"), header=True, inferSchema=True)

In [4]:
movies_df.show(2)

+---------+--------------------+---------+----+-------------+--------+--------------+--------------------+--------------------+------------+--------------------+--------------------+
|   tconst|        primaryTitle|startYear|rank|averageRating|numVotes|runtimeMinutes|           directors|             writers|      genres|            IMDbLink|     Title_IMDb_Link|
+---------+--------------------+---------+----+-------------+--------+--------------+--------------------+--------------------+------------+--------------------+--------------------+
|tt0111161|The Shawshank Red...|     1994|   1|          9.3| 3246333|           142|      Frank Darabont|Stephen King, Fra...|       Drama|"<a href=""https:...|"<a href=""https:...|
|tt0068646|       The Godfather|     1972|   2|          9.2| 2261044|           175|Francis Ford Coppola|Mario Puzo, Franc...|Crime, Drama|"<a href=""https:...|"<a href=""https:...|
+---------+--------------------+---------+----+-------------+--------+--------------+

In [5]:
movies_df.columns

['tconst',
 'primaryTitle',
 'startYear',
 'rank',
 'averageRating',
 'numVotes',
 'runtimeMinutes',
 'directors',
 'writers',
 'genres',
 'IMDbLink',
 'Title_IMDb_Link']

## Mount Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
LOCAL_DB='/content/movies.duckdb'
DB_PATH= "/content/drive/MyDrive/data_engineering/movies.duckdb"

### Saving extracted data into bronze tables

In [8]:
import duckdb

In [9]:
con= duckdb.connect(LOCAL_DB)

In [10]:
con.register(f"movies_bronze_tmp",movies_df.toPandas())
con.execute("""
  CREATE OR REPLACE TABLE movies_bronze AS
  SELECT * FROM movies_bronze_tmp
""")
con.unregister("movies_bronze_tmp")

In [11]:
con.execute("""
SELECT *
FROM movies_bronze
LIMIT 2
""").fetchdf()

,tconst,primaryTitle,startYear,rank,averageRating,numVotes,runtimeMinutes,directors,writers,genres,IMDbLink,Title_IMDb_Link
0,tt0111161,The Shawshank Redemption,1994,1,9.3,3246333,142,Frank Darabont,"Stephen King, Frank Darabont",Drama,"""<a href=""""https://www.imdb.com/title/tt011116...","""<a href=""""https://www.imdb.com/title/tt011116..."
1,tt0068646,The Godfather,1972,2,9.2,2261044,175,Francis Ford Coppola,"Mario Puzo, Francis Ford Coppola","Crime, Drama","""<a href=""""https://www.imdb.com/title/tt006864...","""<a href=""""https://www.imdb.com/title/tt006864..."


In [12]:
print(con.execute("""SHOW TABLES""").fetchdf())

            name
0  movies_bronze


In [13]:
print(con.execute("""
  SELECT COUNT(*) AS row_count
  FROM movies_bronze
""").fetchdf())

   row_count
0       5000


In [14]:
con.close()

In [15]:
shutil.copy2(
    LOCAL_DB,
    DB_PATH
)

'/content/drive/MyDrive/data_engineering/movies.duckdb'

In [16]:
print("Local DB size:",
      os.path.getsize(LOCAL_DB))

print("Drive DB size:",
      os.path.getsize(DB_PATH))

Local DB size: 1060864
Drive DB size: 1060864
